In [0]:
%sql
    
-- ===========================================
-- Quantidade de avaliações por seller 28,56,365 dias e lifetime
-- ===========================================

WITH tb_pedidos AS (
    SELECT *
    FROM workspace.olist.orders
    WHERE order_purchase_timestamp < '2018-07-01'
),

base AS (
    SELECT 
        oi.seller_id, 
        o.order_id,
        DATE(r.review_creation_date) AS review_date
    FROM (
        SELECT seller_id, order_id
        FROM olist.order_items
    ) oi
    JOIN tb_pedidos o 
        ON oi.order_id = o.order_id
    JOIN olist.order_reviews r 
        ON o.order_id = r.order_id
    WHERE r.review_creation_date IS NOT NULL
)

SELECT
    b.seller_id,

    -- ✅ 28 dias
    COUNT(DISTINCT CASE 
        WHEN b.review_date BETWEEN DATE_SUB('2018-07-02', 28) AND '2018-07-02'
        THEN b.order_id 
    END) AS reviews_28d,

    -- ✅ 56 dias
    COUNT(DISTINCT CASE 
        WHEN b.review_date BETWEEN DATE_SUB('2018-07-02', 56) AND '2018-07-02'
        THEN b.order_id 
    END) AS reviews_56d,

    -- ✅ 365 dias
    COUNT(DISTINCT CASE 
        WHEN b.review_date BETWEEN DATE_SUB('2018-07-02', 365) AND '2018-07-02'
        THEN b.order_id 
    END) AS reviews_365d,

    -- ✅ lifetime
    COUNT(DISTINCT b.order_id) AS reviews_lifetime

FROM base b
GROUP BY b.seller_id
ORDER BY reviews_lifetime DESC;

In [0]:
%sql
-- ===========================================
-- Percentual de avaliações por seller 28,56,365 dias e lifetime
-- ===========================================

WITH tb_pedidos AS (
    SELECT *
    FROM workspace.olist.orders
    WHERE order_purchase_timestamp < '2018-07-01'
),

-- ===========================================
-- RELAÇÃO ÚNICA SELLER x ORDER
-- ===========================================
seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        o.order_id,
        CAST(o.order_purchase_timestamp AS DATE) AS order_date
    FROM olist.order_items oi
    JOIN tb_pedidos o
        ON oi.order_id = o.order_id
),

-- ===========================================
-- FLAG DE REVIEW (AGORA COM SELLER)
-- ===========================================
orders_flag AS (
    SELECT
        so.seller_id,
        so.order_id,
        so.order_date,
        CASE 
            WHEN r.order_id IS NOT NULL THEN 1
            ELSE 0
        END AS has_review
    FROM seller_orders so
    LEFT JOIN (
        SELECT DISTINCT order_id
        FROM olist.order_reviews
    ) r
        ON so.order_id = r.order_id
)

-- ===========================================
-- RESULTADO FINAL (POR SELLER)
-- ===========================================
SELECT

    f.seller_id,

    -- ======================
    -- 28 DIAS
    -- ======================
    COUNT(DISTINCT CASE
        WHEN f.order_date >= date_sub('2018-07-01', 27)
        THEN f.order_id
    END) AS pedidos_28d,

    COUNT(DISTINCT CASE
        WHEN f.order_date >= date_sub('2018-07-01', 27)
             AND f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_28d,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.order_date >= date_sub('2018-07-01', 27)
                 AND f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT CASE
            WHEN f.order_date >= date_sub('2018-07-01', 27)
            THEN f.order_id
        END), 0)
    , 2) AS percentual_28d,

    -- ======================
    -- 56 DIAS
    -- ======================
    COUNT(DISTINCT CASE
        WHEN f.order_date >= date_sub('2018-07-01', 55)
        THEN f.order_id
    END) AS pedidos_56d,

    COUNT(DISTINCT CASE
        WHEN f.order_date >= date_sub('2018-07-01', 55)
             AND f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_56d,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.order_date >= date_sub('2018-07-01', 55)
                 AND f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT CASE
            WHEN f.order_date >= date_sub('2018-07-01', 55)
            THEN f.order_id
        END), 0)
    , 2) AS percentual_56d,

    -- ======================
    -- 365 DIAS
    -- ======================
    COUNT(DISTINCT CASE
        WHEN f.order_date >= date_sub('2018-07-01', 364)
        THEN f.order_id
    END) AS pedidos_365d,

    COUNT(DISTINCT CASE
        WHEN f.order_date >= date_sub('2018-07-01', 364)
             AND f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_365d,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.order_date >= date_sub('2018-07-01', 364)
                 AND f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT CASE
            WHEN f.order_date >= date_sub('2018-07-01', 364)
            THEN f.order_id
        END), 0)
    , 2) AS percentual_365d,

    -- ======================
    -- LIFETIME
    -- ======================
    COUNT(DISTINCT f.order_id) AS pedidos_lifetime,

    COUNT(DISTINCT CASE
        WHEN f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_lifetime,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT f.order_id), 0)
    , 2) AS percentual_lifetime

FROM orders_flag f
GROUP BY f.seller_id
ORDER BY percentual_lifetime DESC;